# AF-Edge V5: 500Hz/30s ECG for TriAnswer → M55M1
**Matches TriAnswer BLE output**: 500Hz, 30s window (15000 samples), stride 10s (5000)
**3-class (N/AF/O)** | **INT8 Ethos-U55** | PhysioNet 300Hz resampled to 500Hz


In [ ]:
# Cell 1: Setup
import os, sys
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q scipy wfdb scikit-learn matplotlib seaborn
import tensorflow as tf, numpy as np
print(f'TF={tf.__version__}, GPU={tf.config.list_physical_devices("GPU")}')
for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)
tf.keras.mixed_precision.set_global_policy('mixed_float16')


In [ ]:
# Cell 2: Upload
import zipfile
from google.colab import files
uploaded = files.upload()
for f in uploaded:
    if f.endswith('.zip'): zipfile.ZipFile(f).extractall('.')
DATA_DIR = REF_FILE = None
for d in ['data/training2017','training2017']:
    if os.path.isdir(d) and any(x.endswith('.mat') for x in os.listdir(d)):
        DATA_DIR=d; break
for f in ['data/REFERENCE.csv','REFERENCE.csv']:
    if os.path.isfile(f): REF_FILE=f; break
assert DATA_DIR and REF_FILE, 'Data not found'
print(f'{DATA_DIR}, {REF_FILE}')


In [ ]:
# Cell 3: Load - Resample 300Hz→500Hz, 30s window/10s stride (TriAnswer format)
import pandas as pd
from scipy import io as sio
from scipy.signal import butter, filtfilt, resample
from collections import Counter
import gc, warnings; warnings.filterwarnings('ignore')

ref = pd.read_csv(REF_FILE, header=None, names=['id','label'])
LABEL_MAP = {'N':0, 'A':1, 'O':2, '~':2}
CLASS_NAMES = ['Normal','AF','Other']
NUM_CLASSES = 3

# === TriAnswer-matching parameters ===
FS_ORIG = 300       # PhysioNet original sampling rate
FS_TARGET = 500     # TriAnswer sampling rate
WIN_SEC = 30        # seconds per window
STRIDE_SEC = 10     # seconds stride
WIN_LEN = FS_TARGET * WIN_SEC    # 15000 samples
STRIDE = FS_TARGET * STRIDE_SEC  # 5000 samples
print(f'Target: {FS_TARGET}Hz, window={WIN_LEN} samples ({WIN_SEC}s), stride={STRIDE} ({STRIDE_SEC}s)')

def bandpass(sig, fs):
    b,a = butter(3, [0.5/(fs/2), min(45, fs/2-1)/(fs/2)], 'band')
    return filtfilt(b,a,sig).astype(np.float32)

def resample_300_to_500(sig):
    """Resample from 300Hz to 500Hz (ratio 5:3)."""
    n_target = int(len(sig) * FS_TARGET / FS_ORIG)
    return resample(sig, n_target).astype(np.float32)

def process_and_extract(sig_300, label):
    """Bandpass at 300Hz, resample to 500Hz, z-score normalize, extract windows."""
    # Bandpass at original rate
    try: sig_300 = bandpass(sig_300, FS_ORIG)
    except: pass
    # Resample 300→500Hz
    sig = resample_300_to_500(sig_300)
    # Z-score normalize (same as MCU will do with raw ADC)
    mu, std = np.mean(sig), np.std(sig)
    sig = (sig-mu)/std if std>1e-6 else sig-mu
    sig = sig.astype(np.float32)
    # Extract windows
    wins, labs = [], []
    if len(sig) < WIN_LEN:
        p = np.zeros(WIN_LEN, dtype=np.float32)
        p[:len(sig)] = sig
        wins.append(p); labs.append(label)
    else:
        s = 0
        while s+WIN_LEN <= len(sig):
            wins.append(sig[s:s+WIN_LEN]); labs.append(label); s+=STRIDE
        if s < len(sig) and len(sig)>=WIN_LEN:
            wins.append(sig[-WIN_LEN:]); labs.append(label)
    return wins, labs

print('Loading & resampling 300Hz→500Hz...')
all_w, all_l, all_r = [], [], []
for i,row in ref.iterrows():
    try:
        sig = sio.loadmat(os.path.join(DATA_DIR,row['id']+'.mat'))['val'].flatten().astype(np.float32)
        if len(sig)<FS_ORIG*5: continue  # min 5s
        w,l = process_and_extract(sig, LABEL_MAP[row['label']])
        all_w.extend(w); all_l.extend(l); all_r.extend([row['id']]*len(w))
    except: continue
    if (i+1)%2000==0: print(f'  {i+1}/{len(ref)}')
X_all = np.array(all_w, dtype=np.float32)[...,np.newaxis]
del all_w; gc.collect()
y_all = np.array(all_l, dtype=np.int32)
rec_ids = np.array(all_r)
print(f'\nWindows: {len(X_all)}, Shape: {X_all.shape}')
print(f'RAM: {X_all.nbytes/1024**3:.2f} GB')
print(f'Dist: {dict(Counter(y_all))}')


In [ ]:
# Cell 4: Record-level split
from sklearn.model_selection import train_test_split
import gc

uids = np.unique(rec_ids)
uid2lab = {r:y_all[rec_ids==r][0] for r in uids}
uid_labs = np.array([uid2lab[r] for r in uids])
tr_ids,te_ids = train_test_split(uids,test_size=0.12,random_state=42,stratify=uid_labs)
tr_ids,va_ids = train_test_split(tr_ids,test_size=0.12,random_state=42,
    stratify=np.array([uid2lab[r] for r in tr_ids]))
tr_m,va_m,te_m = np.isin(rec_ids,tr_ids),np.isin(rec_ids,va_ids),np.isin(rec_ids,te_ids)
X_train,y_train = X_all[tr_m],y_all[tr_m]
X_val,y_val = X_all[va_m],y_all[va_m]
X_test,y_test = X_all[te_m],y_all[te_m]
del X_all, y_all; gc.collect()
print(f'Train:{len(X_train)} Val:{len(X_val)} Test:{len(X_test)}')
print(f'Train dist: {dict(Counter(y_train))}')
print(f'RAM: {(X_train.nbytes+X_val.nbytes+X_test.nbytes)/1024**3:.2f} GB')


In [ ]:
# Cell 5: Model for 15000-sample input (30s@500Hz)
from tensorflow.keras import layers, Model, regularizers

REG = 0.002

def cb(x, f, k, s=1):
    x = layers.Conv1D(f,k,strides=s,padding='same',use_bias=False,
                      kernel_regularizer=regularizers.l2(REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def res(x, f, k, s=1):
    sc = x
    o = cb(x,f,k,s)
    o = layers.Conv1D(f,k,padding='same',use_bias=False,
                      kernel_regularizer=regularizers.l2(REG))(o)
    o = layers.BatchNormalization()(o)
    if s>1 or sc.shape[-1]!=f:
        sc = layers.Conv1D(f,1,strides=s,padding='same',use_bias=False)(sc)
        sc = layers.BatchNormalization()(sc)
    o = layers.Add()([o,sc])
    o = layers.ReLU()(o)
    return o

def build_model(input_shape=(15000,1), nc=3):
    inp = layers.Input(shape=input_shape, name='ecg_input')
    # Multi-scale stem
    b1 = cb(inp, 20, 7)     # Fine features
    b2 = cb(inp, 20, 25)    # QRS complex (~50ms@500Hz)
    b3 = cb(inp, 12, 75)    # RR interval (~150ms)
    x = layers.Concatenate()([b1,b2,b3])  # (15000, 52)
    x = layers.MaxPooling1D(5)(x)          # (3000, 52)
    x = layers.SpatialDropout1D(0.15)(x)
    # ResBlocks - progressive downsampling
    x = res(x, 52, 7, 2)    # 1500
    x = layers.SpatialDropout1D(0.15)(x)
    x = res(x, 64, 7, 2)    # 750
    x = layers.SpatialDropout1D(0.2)(x)
    x = res(x, 80, 5, 2)    # 375
    x = layers.SpatialDropout1D(0.2)(x)
    x = res(x, 96, 5, 2)    # 188
    x = layers.SpatialDropout1D(0.25)(x)
    x = res(x, 96, 3, 2)    # 94
    x = layers.SpatialDropout1D(0.25)(x)
    x = res(x, 96, 3, 2)    # 47
    # Head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, kernel_regularizer=regularizers.l2(REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(nc, dtype='float32')(x)
    out = layers.Activation('softmax', dtype='float32', name='output')(x)
    return Model(inp, out, name='AF_Edge_v5')

model = build_model()
model.summary()
p = model.count_params()
print(f'\nParams: {p:,} | INT8 est: {p/1024:.0f}KB')


In [ ]:
# Cell 6: Training
import math, numpy as np, tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

cw_vals = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weights = {i:w for i,w in enumerate(cw_vals)}
print(f'Class weights: {class_weights}')

BATCH = 64  # Smaller batch for 15000-len windows (more GPU memory per sample)

def tf_augment(x, y):
    shift = tf.random.uniform([], -1000, 1000, dtype=tf.int32)  # ±2s@500Hz
    x = tf.roll(x, shift, axis=0)
    x = x * tf.random.uniform([], 0.85, 1.15)
    x = x + tf.cast(tf.random.normal(tf.shape(x), 0, 0.03), x.dtype)
    return x, y

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(len(X_train))
    .map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH).prefetch(tf.data.AUTOTUNE))
val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(128).prefetch(tf.data.AUTOTUNE))

EPOCHS = 200
MAX_LR = 1e-3
MIN_LR = 1e-6
WARMUP = 8

def cosine_lr(epoch, lr):
    if epoch < WARMUP: return MAX_LR*(epoch+1)/WARMUP
    p = (epoch-WARMUP)/(EPOCHS-WARMUP)
    return MIN_LR+0.5*(MAX_LR-MIN_LR)*(1+math.cos(math.pi*p))

model.compile(
    optimizer=tf.keras.optimizers.AdamW(MAX_LR, weight_decay=5e-4, clipnorm=1.0),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint('best_model.keras',
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=40,
        restore_best_weights=True, mode='max', verbose=1),
    tf.keras.callbacks.LearningRateScheduler(cosine_lr),
]

print(f'Training: {EPOCHS} epochs, batch={BATCH}, input_shape={X_train.shape[1:]}')
history = model.fit(train_ds, validation_data=val_ds,
    epochs=EPOCHS, class_weight=class_weights,
    callbacks=callbacks, verbose=1)


In [ ]:
# Cell 7: Evaluation + TTA + Recording-level
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

model = tf.keras.models.load_model('best_model.keras')

y_prob_raw = model.predict(X_test, batch_size=128)
y_pred_raw = np.argmax(y_prob_raw, axis=1)
acc_raw = accuracy_score(y_test, y_pred_raw)
print(f'Single model window acc: {acc_raw*100:.2f}%')

# TTA
print('Running TTA...')
all_preds = [y_prob_raw]
for _ in range(5):
    X_aug = X_test.copy()
    for i in range(len(X_aug)):
        X_aug[i] = np.roll(X_aug[i], np.random.randint(-500,500), axis=0)
        X_aug[i] *= np.random.uniform(0.9, 1.1)
        X_aug[i] += np.random.normal(0, 0.02, X_aug[i].shape)
    all_preds.append(model.predict(X_aug, batch_size=128, verbose=0))
y_prob = np.mean(all_preds, axis=0)
y_pred = np.argmax(y_prob, axis=1)
acc_tta = accuracy_score(y_test, y_pred)
print(f'TTA window acc: {acc_tta*100:.2f}%')

# Recording-level
test_rids = rec_ids[te_m]
rp, rt = {}, {}
for i,r in enumerate(test_rids):
    rp.setdefault(r,[]).append(y_prob[i])
    rt[r] = y_test[i]
rl_pred = np.array([np.argmax(np.mean(rp[r],axis=0)) for r in rp])
rl_true = np.array([rt[r] for r in rp])
acc_rec = accuracy_score(rl_true, rl_pred)
f1_rec = f1_score(rl_true, rl_pred, average='macro')
print(f'\nRecording-level acc: {acc_rec*100:.2f}%')
print(f'{classification_report(rl_true, rl_pred, target_names=CLASS_NAMES)}')

fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
a1.plot(history.history['accuracy'],label='Train')
a1.plot(history.history['val_accuracy'],label='Val')
a1.set_title('Accuracy');a1.legend();a1.grid(True)
a2.plot(history.history['loss'],label='Train')
a2.plot(history.history['val_loss'],label='Val')
a2.set_title('Loss');a2.legend();a2.grid(True)
plt.tight_layout();plt.show()

cm=confusion_matrix(rl_true,rl_pred)
plt.figure(figsize=(7,5))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
            xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES)
plt.title(f'Recording-Level (Acc={acc_rec*100:.1f}%)')
plt.tight_layout();plt.show()


In [ ]:
# Cell 8: Pruning
def prune(mdl, sp):
    for layer in mdl.layers:
        ws = layer.get_weights()
        nw = []
        for w in ws:
            if w.ndim>=2:
                th = np.percentile(np.abs(w), sp*100)
                nw.append(w * (np.abs(w)>=th))
            else: nw.append(w)
        if nw: layer.set_weights(nw)

model_p = build_model()
model_p.set_weights(model.get_weights())
ft_ds = tf.data.Dataset.from_tensor_slices((X_train,y_train)).shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)
ft_val = tf.data.Dataset.from_tensor_slices((X_val,y_val)).batch(128).prefetch(tf.data.AUTOTUNE)

for sp in [0.2, 0.4]:
    print(f'Pruning {sp*100:.0f}%...')
    prune(model_p, sp)
    model_p.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
        loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model_p.fit(ft_ds, validation_data=ft_val, epochs=8, verbose=1)

yp = np.argmax(model_p.predict(X_test,batch_size=128),axis=1)
acc_p = accuracy_score(y_test,yp)
print(f'Pruned acc: {acc_p*100:.2f}%')
model_p.save('pruned.keras')
model_stripped = model_p


In [ ]:
# Cell 9: INT8 Quantization
tf.keras.mixed_precision.set_global_policy('float32')
model_fp32 = build_model()
model_fp32.set_weights(model_stripped.get_weights())
model_fp32.export('saved_model')

def rep_data():
    idx = np.random.choice(len(X_train), min(300,len(X_train)), replace=False)
    for i in idx: yield [X_train[i:i+1].astype(np.float32)]

c = tf.lite.TFLiteConverter.from_saved_model('saved_model')
c.optimizations = [tf.lite.Optimize.DEFAULT]
c.representative_dataset = rep_data
c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
c.inference_input_type = tf.int8
c.inference_output_type = tf.int8
tflite = c.convert()
with open('model_int8.tflite','wb') as f: f.write(tflite)
kb = os.path.getsize('model_int8.tflite')/1024
print(f'INT8: {kb:.1f}KB | Flash:{"OK" if kb<1024 else "!"} | SRAM:{"OK" if kb<512 else "?"}')


In [ ]:
# Cell 10: Validate INT8
interp = tf.lite.Interpreter(model_path='model_int8.tflite')
interp.allocate_tensors()
id_,od_ = interp.get_input_details()[0], interp.get_output_details()[0]
isc,izp = id_['quantization']
osc,ozp = od_['quantization']
print(f'Input shape: {id_["shape"]}')  # Should be [1, 15000, 1]
print(f'In: scale={isc} zp={izp} | Out: scale={osc} zp={ozp}')

yq = []
for i in range(len(X_test)):
    xq = (X_test[i:i+1]/isc+izp).astype(np.int8)
    interp.set_tensor(id_['index'],xq)
    interp.invoke()
    yq.append(np.argmax(interp.get_tensor(od_['index']),axis=1)[0])
yq = np.array(yq)
aq = accuracy_score(y_test,yq)
print(f'\n=== Results ===')
print(f'Single={acc_raw*100:.2f}% TTA={acc_tta*100:.2f}% Rec={acc_rec*100:.2f}%')
print(f'Pruned={acc_p*100:.2f}% INT8={aq*100:.2f}%')
print(f'\n{classification_report(y_test,yq,target_names=CLASS_NAMES)}')


In [ ]:
# Cell 11: Vela + C Header for M55M1
!pip install -q ethos-u-vela
!vela model_int8.tflite --accelerator-config ethos-u55-256 --optimise Performance --output-dir vela_out
import glob
vf = glob.glob('vela_out/*.tflite')
vp = vf[0] if vf else 'model_int8.tflite'
print(f'Vela: {os.path.getsize(vp)/1024:.1f}KB')
with open(vp,'rb') as f: d=f.read()
ca=','.join(f'0x{b:02x}' for b in d)
with open('af_model.h','w') as f:
    f.write(f'// AF-Edge V5 for M55M1 (Cortex-M55+Ethos-U55)\n'
            f'// Input: 15000 x int8 (30s @ 500Hz, z-score normalized)\n'
            f'// Output: 3 x int8 (Normal/AF/Other)\n'
            f'// TriAnswer BLE → z-score → int8 quantize → inference\n'
            f'// Size: {len(d)} bytes ({len(d)/1024:.1f} KB)\n'
            f'#ifndef AF_MODEL_H\n#define AF_MODEL_H\n#include <stdint.h>\n'
            f'const unsigned int af_model_len={len(d)};\n'
            f'alignas(16) const uint8_t af_model_data[]={{\n {ca}\n}};\n'
            f'// Quantization: in_scale={isc} in_zp={izp} out_scale={osc} out_zp={ozp}\n'
            f'// Preprocessing on MCU:\n'
            f'//   1. Collect 15000 samples (30s) from TriAnswer BLE at 500Hz\n'
            f'//   2. Z-score: x_norm = (x - mean) / std\n'
            f'//   3. Quantize: x_int8 = (int8_t)(x_norm / {isc} + {izp})\n'
            f'//   4. Run inference\n'
            f'//   5. Output argmax: 0=Normal 1=AF 2=Other\n'
            f'#endif\n')
print('C header: af_model.h')


In [ ]:
# Cell 12: Download
import shutil
os.makedirs('deploy',exist_ok=True)
shutil.copy('model_int8.tflite','deploy/')
if os.path.exists('af_model.h'): shutil.copy('af_model.h','deploy/')
for v in glob.glob('vela_out/*.tflite'): shutil.copy(v,'deploy/')
shutil.make_archive('deploy','zip','.','deploy')
print(f'Single={acc_raw*100:.2f}% TTA={acc_tta*100:.2f}% Rec={acc_rec*100:.2f}% INT8={aq*100:.2f}%')
print(f'Model: {kb:.1f}KB | Input: 15000x1 (30s@500Hz) | Classes: {CLASS_NAMES}')
from google.colab import files
files.download('deploy.zip')
